# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question.** A content team can review about 50 pages a month out of thousands. Which 50 should a person open first?

**Decision it supports.** Ordering a monthly human review of pages (lane: refresh / content opportunity scoring). It is a ranking problem judged at a fixed review budget, **precision@50**, always shown next to the **base rate** (what random picking gives).

**The target is an observed proxy.** A page is *declining* when its next-month clicks are below 80% of its latest month. It does not say why, and it does not say a refresh would help.

**This notebook mirrors the deployed paper.** It reads the receipts committed in `work/outputs/` (data contract, baseline, model, validation audit, playbook), prints the evidence behind every section, and in section 7 regenerates the paper's figures and `docs/index.html`.

In [1]:
# Capstone setup: load the committed receipts. Every number in the paper traces back to one of these five files.
import calendar, datetime, json, os, shutil, urllib.request
import numpy as np, pandas as pd

RAW = "https://raw.githubusercontent.com/HamzaMohamed25/flyrank-ml-internship/main/work/outputs/"
NEEDED = ["data_contract_v2.json", "baseline_metrics.json", "w05_model_metrics.json", "w06_validation_metrics.json", "w07_playbook_metrics.json"]

def load_receipt(name):
    for path in (f"work/outputs/{name}",):
        if os.path.exists(path):
            return json.load(open(path))
    try:                                                    # the public repo
        with urllib.request.urlopen(RAW + name, timeout=30) as r:
            return json.loads(r.read())
    except Exception as e:
        print(f"{name}: not in this session and not readable from the repo ({type(e).__name__}).")
    from google.colab import files                          # last resort: upload it from your computer
    print(f"Please choose {name} from your computer.")
    up = files.upload()
    return json.loads(next(iter(up.values())))

R = {n: load_receipt(n) for n in NEEDED}
dc, bl, w5, w6, w7 = (R[n] for n in NEEDED)

# A few numbers that were printed by earlier notebooks but were not saved in a receipt (source noted for each).
FACTS = {
    "apr_pages": 16778, "apr_clients": 29, "apr_base": 0.538,                                   # ML-09 output, 1 April decision point
    "jun_clients": 37,                                                                          # ML-10 output, 1 June decision point
    "queue_clients": 8, "queue_top_client_share": 0.32, "queue_median_clicks": 15,              # ML-10 output, the 50-page queue
    "err_hits": 126, "err_total": 250, "err_missed_low": 1205, "err_declines": 6043, "err_top_quarter": 1814,   # ML-08 output
}
print("Receipts loaded:")
print(pd.DataFrame({"receipt": NEEDED, "top-level keys": [len(R[n]) for n in NEEDED]}).to_string(index=False))
print(f"\nContract: {dc['rows']:,} eligible pages of {dc['rows_before_eligibility']:,} | {dc['clients']} clients | release {dc['release']} | seed {w5['seed']}")
print("Versions used:", w5["versions"])

w06_validation_metrics.json: not in this session and not readable from the repo (HTTPError).
Please choose w06_validation_metrics.json from your computer.


Saving w06_validation_audit (1).ipynb to w06_validation_audit (1).ipynb
Receipts loaded:
                    receipt  top-level keys
      data_contract_v2.json              16
      baseline_metrics.json              18
     w05_model_metrics.json              13
w06_validation_metrics.json               4
  w07_playbook_metrics.json              16

Contract: 16,513 eligible pages of 380,147 | 36 clients | release v20260703 | seed 42
Versions used: {'python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit-learn': '1.6.1'}


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

- **Release and table:** FlyRank ML Internship warehouse, release `v20260703`; only the daily page-performance table, only clicks, impressions and average position, aggregated to page-level months. Identifiers are hashed.
- **Three decision points:** 1 April (Jan-Mar -> April), 1 May (Feb-Apr -> May), and a sealed 1 June (Mar-May -> June) that was read once after everything was frozen.
- **Eligibility:** at least 1,000 impressions over the three feature months and at least 10 clicks in the latest one: 16,513 of 380,147 pages from 36 clients at the May point.
- **Excluded:** product flags and health scores (never loaded), session / scroll / AI-referral fields (sparse), and a block of 833 identical pages from one client (99.6% declined; 173 at the June point). Pages with new or thin history are scored but are not eligible for the queue.
- **New in this notebook:** a coverage check of every month, because the label rate jumped between May and June and a month with missing days would make clicks look lower.

In [2]:
# Data: coverage of each month in the warehouse (is every label month complete?) and the three decision points
def hf_token():
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

COV = {}
try:
    for m in ["2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]:
        d = pd.read_parquet(f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={m}/", columns=["report_date"], storage_options={"token": hf_token()})
        dates = pd.to_datetime(d["report_date"]); y, mm = int(m[:4]), int(m[5:])
        COV[m] = {"days_with_data": int(dates.nunique()), "days_in_month": calendar.monthrange(y, mm)[1], "first_day": str(dates.min().date()), "last_day": str(dates.max().date()), "daily_rows": int(len(d))}
        del d, dates
    cov = pd.DataFrame(COV).T; cov["rows_per_day"] = (cov["daily_rows"].astype(int) / cov["days_with_data"].astype(int)).round(0).astype(int)
    print("Coverage of each month in the warehouse (report_date only):"); print(cov.to_string())
    os.makedirs("work/outputs", exist_ok=True); json.dump(COV, open("work/outputs/capstone_data_coverage.json", "w"), indent=2)
    incomplete = [m for m, v in COV.items() if v["days_with_data"] < v["days_in_month"]]
    print("\nMonths with fewer days than the calendar month:", incomplete or "none")
except Exception as e:
    COV = {}
    print("Coverage check skipped (the warehouse could not be read here):", type(e).__name__, str(e)[:120])

dp = pd.DataFrame([
    {"decision point": "1 April", "features": "Jan-Mar", "label month": "April", "pages": FACTS["apr_pages"], "clients": FACTS["apr_clients"], "base rate": FACTS["apr_base"]},
    {"decision point": "1 May", "features": "Feb-Apr", "label month": "May", "pages": bl["rows_analysis_frame"], "clients": dc["clients"], "base rate": round(bl["base_rate"], 3)},
    {"decision point": "1 June (sealed)", "features": "Mar-May", "label month": "June", "pages": w7["scoring_pages"], "clients": FACTS["jun_clients"], "base rate": round(w7["sealed_june_evaluation"]["all contract pages"]["base_rate"], 3)}])
print("\nThe three decision points (after the identical-row block is excluded):"); print(dp.to_string(index=False))
print(f"\nIdentical-row block excluded: {bl['templated_block']['rows_excluded']:,} pages at the May point (decline rate {bl['templated_block']['decline_rate']:.3f}), {w7['identical_block_excluded']['scoring']:,} at the June point.")

Coverage of each month in the warehouse (report_date only):
        days_with_data days_in_month   first_day    last_day daily_rows  rows_per_day
2026-01             31            31  2026-01-01  2026-01-31    7890817        254542
2026-02             28            28  2026-02-01  2026-02-28    7355108        262682
2026-03             31            31  2026-03-01  2026-03-31    9841378        317464
2026-04             30            30  2026-04-01  2026-04-30   10424730        347491
2026-05             31            31  2026-05-01  2026-05-31   11687376        377012
2026-06             30            30  2026-06-01  2026-06-30   11694072        389802

Months with fewer days than the calendar month: none

The three decision points (after the identical-row block is excluded):
 decision point features label month  pages  clients  base rate
        1 April  Jan-Mar       April  16778       29      0.538
          1 May  Feb-Apr         May  15680       36      0.385
1 June (sealed)  Mar

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

- **Label:** next-month clicks below 80% of the latest month. **Features:** nine, all known at the decision point (window and latest-month impressions and clicks, log-momentum, CTR, impression-weighted position, active days).
- **Baseline:** the frozen ML-07 rule (latest month at least 2x the earlier average, with at least 10 clicks in each earlier month), written before any model.
- **Models:** Logistic Regression, depth-4 tree, Random Forest (declared primary in advance), small Hist Gradient Boosting; fixed settings, no tuning on any held-out fold; seed 42.
- **Validation ladder:** random rows (contrast) -> client-grouped folds -> time only -> client + time -> a sealed month read once.
- **Leakage checks:** features end before the label window; the label ingredient is never a feature; a deliberate leak (harness) pushes precision@50 to 1.00 and AUC to 1.000, proving the audit can see a leak; feature-group ablation.
- **Disclosed choices made after looking at results:** rule direction, history guard, identical-block exclusion, and the Logistic Regression highlight (which is why a sealed month was added).

In [5]:
# Methodology: the label, features, baseline, models and every setting frozen before the sealed month, straight from the receipts
print("Label: next-month clicks < 80% of the latest month (observed proxy). Features (9, all known at the decision point):")
print("  ", dc["features"])
print(f"\nBaseline (frozen rule): {bl['rule']}")
print(f"  history guard: >= {bl['history_floor']} clicks in each earlier month | ratio threshold {bl['min_ratio']}x | tie-break: {bl['tie_break']}")
print("\nModels and fixed settings (ML-08, no tuning on any held-out fold):")
for k, v in w5["models"].items():
    print(f"  {k}: {v[:150]}")
print(f"\nValidation designs: {w5['split']} | random rows (contrast) | time-forward 1 Apr -> 1 May | client + time | sealed June read once")
print("Frozen before June was read:"); print(" ", json.dumps(w7["frozen_before_june"]))
# print(f"\nLeak harness (a column built from the label ingredient, added on purpose): P@50 = {w6['leak_harness']['P@50']:.2f}, AUC = {w6['leak_harness']['AUC']:.3f}")
print("Disclosed post-hoc choices:", "; ".join(bl["disclosures"]), "; model choice (Logistic Regression) after seeing five folds, which is why a sealed month was added.")

Label: next-month clicks < 80% of the latest month (observed proxy). Features (9, all known at the decision point):
   ['impressions_total', 'clicks_total', 'impressions_apr', 'clicks_apr', 'clicks_feb', 'momentum_apr_vs_feb', 'ctr', 'weighted_position', 'active_days']

Baseline (frozen rule): April clicks >= 2x Feb-Mar average, with >=10 clicks in Feb and in Mar; score = 100 * min(log2(ratio)/3, 1) * impressions_pctile_apr
  history guard: >= 10 clicks in each earlier month | ratio threshold 2.0x | tie-break: hash(content_hash_id|42)

Models and fixed settings (ML-08, no tuning on any held-out fold):
  Logistic Regression: {'memory': None, 'steps': [('columntransformer', ColumnTransformer(transformers=[('counts',
                                 Pipeline(steps=[('functio
  Decision Tree (depth 4): {'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 4, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0
  Random Forest: {'bootstrap': True, 'c

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Precision@50, mean over folds, with the base rate beside it (the cells below print the full tables from the receipts):

| Design | Frozen rule | Logistic Regression | Random Forest | Base rate |
|---|---|---|---|---|
| Random rows | 0.37 | 0.58 | **0.73** | 0.385 |
| Client-grouped folds | 0.36 | 0.62 | 0.50 | 0.391 |
| Time only (Apr to May) | 0.42 | 0.58 | 0.50 | 0.385 |
| Client + time (Apr to May) | 0.36 | 0.54 | 0.56 | 0.391 |

**Sealed June month** (base rate 0.652 on all pages, up from 0.385 in May): on the queue population the model's top 50 scored 0.82 against a 0.70 base rate and the rule scored 0.46; across all contract pages the model scored 0.62 against 0.65. AUC is 0.53 to 0.55.

**Honest reading (observed, directional):** a random split inflated the Random Forest by about 23 points; on unseen clients and a later month both models sit about 15 points above random picking and the rule does not; in the sealed month the lift is population-dependent, the surge pattern reversed, and the base-rate check fired.

In [6]:
# Results: model vs baseline on the same rows, folds, metric and K, with the base rate always beside it
L6 = pd.DataFrame(w6["design_ladder"]).rename(columns={"P@50": "precision@50"})
print("1) The validation ladder (mean over folds; base rate = random picking):")
print(L6[["design", "method", "precision@50", "AUC", "base rate", "P@50 sd across folds"]].round(3).to_string(index=False))

pf = pd.DataFrame(w5["per_fold"])
s5 = pf.groupby("method").agg(p10=("p10", "mean"), p50=("p50", "mean"), p100=("p100", "mean"), auc=("auc", "mean"), p50_sd=("p50", "std")).round(3)
print(f"\n2) Client-grouped folds, five methods (ML-08; mean base rate {pf.groupby('fold')['base_rate'].first().mean():.3f}):"); print(s5.to_string())

ev = w7["sealed_june_evaluation"]
for pop, e in ev.items():
    print(f"\n3) Sealed June month, {pop}: {e['pages']:,} pages, base rate {e['base_rate']:.3f}")
    print(pd.DataFrame({m: {"p10": e[m]["p10"], "p50": e[m]["p50"], "p100": e[m]["p100"], "auc": e[m]["auc"]} for m in ("Logistic Regression", "Frozen rule")}).T.round(3).to_string())

at = pd.DataFrame(w7["archetypes_dev"]).merge(pd.DataFrame(w7["archetypes_june"]), on="archetype", suffixes=(" May", " June")).set_index("archetype")
print(f"\n4) Decline rate by archetype, May vs sealed June (base rates {bl['base_rate']:.3f} and {ev['all contract pages']['base_rate']:.3f}):"); print(at.to_string())
print("\n5) Feature-group ablation (client-grouped):"); print(pd.DataFrame(w6["feature_ablation"]).round(3).to_string(index=False))
print("\n6) Page-size bands (top 20% of out-of-fold scores):"); print(pd.DataFrame(w6["size_strata"]).to_string(index=False))

KeyError: 'design_ladder'

## 5. Limitations

*What this work cannot claim.*

- Three one-month windows are thin temporal evidence, not a walk-forward over many months.
- The ranking is weak overall (AUC about 0.55 to 0.58), with 7 to 13 points of spread across folds.
- The label rate was 0.54, 0.39 and 0.65 at the three decision points. Whether the June jump is real, seasonal or a data artifact is not established here (see the coverage check). Scores are not calibrated probabilities.
- The conclusion depends on the population: above the base rate on queue-eligible pages in June, not on all pages.
- The result leans on click-level features that sit close to the label, and the broad ranking adds little inside page-size bands.
- The label is a proxy and partly reflects regression to the mean; it cannot see slow decay.
- The queue is made of small pages (median 15 clicks), 41 of 50 rows are low confidence, and only 1 of 50 has a rule-based reason code.
- No causal claim: nothing here shows that refreshing recovers traffic or explains how a search engine ranks pages.

In [ ]:
# Limitations in numbers
ev = w7["sealed_june_evaluation"]; b_jun = ev["all contract pages"]["base_rate"]
print(f"Base rate by decision point: April {FACTS['apr_base']:.3f} | May {bl['base_rate']:.3f} | June {b_jun:.3f}  (June minus May: {b_jun - bl['base_rate']:+.3f})")
print("Whether the June shift is real, seasonal or a coverage artifact is not established by this data. Coverage by month:",
      {m: f"{v['days_with_data']}/{v['days_in_month']} days" for m, v in COV.items()} if COV else "not available in this session")
print(f"AUC under the honest designs: {pd.DataFrame(w6['design_ladder'])['AUC'].min():.2f} to {pd.DataFrame(w6['design_ladder'])['AUC'].max():.2f} (the frozen rule is about 0.50 everywhere)")
print(f"Queue: {FACTS['queue_clients']} clients, largest client share {FACTS['queue_top_client_share']:.0%}, median clicks in the latest month {FACTS['queue_median_clicks']}, low-confidence rows {w7['queue_confidence'].get('low', 0)} of {sum(w7['queue_confidence'].values())}")
print(f"Reason codes explain only {w7['queue_action_mix'].get('verify_spike_then_review', 0)} of {sum(w7['queue_action_mix'].values())} queue rows; the rest are model-only (investigate_quiet_risk).")
print(f"Pages with new or thin history (not eligible for the queue): {pd.DataFrame(w7['archetypes_june']).set_index('archetype').loc['1 new or thin history', 'pages']:,} of {w7['scoring_pages']:,}")

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**The rule supplies the reason, the model supplies the order, a person decides.** Actions: `verify_spike_then_review` (rule flag), `investigate_quiet_risk` (model top 50 with no rule flag), `monitor_only`, and *not on the queue* (thin history). Every row carries a confidence note (never *high*). A person checks for tracking or campaign artifacts, seasonality, durable climbs, search intent, recent optimization, and tiny counts before acting.

**Not automated:** edits or rewrites, thin-history pages in the queue, identical-row blocks, jumps above 7x without verification, a model score as a reason, calibrated-probability claims, and any promise that a refresh recovers traffic. The recent-optimization cooldown is **not implemented** (it needs optimization history).

**Monitoring on the sealed month:** the base-rate check fired (+0.32 points); population and feature-drift checks did not.

In [ ]:
# Ranked recommendations: the playbook as it was built and tested (ML-10)
print("Queue action mix:", w7["queue_action_mix"], "| confidence:", w7["queue_confidence"])
print("\nNo-go list and how far each rule is implemented:"); print(pd.DataFrame(w7["gates"]).to_string(index=False))
print("\nMonitoring triggers, evaluated on the sealed June month:"); print(pd.DataFrame(w7["triggers"]).to_string(index=False))
print("\nPre-release feature drift (PSI, 1 May training vs 1 June scoring):"); print(pd.DataFrame(w7["psi"]).to_string(index=False))

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The cell below writes four figures to `docs/img/` (and a copy to `work/figures/`) and builds `docs/index.html`, the deployed paper with all nine sections. It first checks that the text still matches the numbers (nine claim checks), then zips the site as `paper_site.zip`.

**To publish:** unzip `paper_site.zip`, upload the `docs` folder to the repo (**Add file -> Upload files**), then **Settings -> Pages -> Deploy from a branch -> `main` -> `/docs`**. Put the page URL on the single line of `submission/paper_url.txt`. Check it in a private window.

In [ ]:
# 7) Build the paper site: figures + docs/index.html. Every number comes from the receipts loaded in the setup cell.
import matplotlib.pyplot as plt
AUTHOR_NAME = "Hamza"                                                       # edit if you want a different byline
REPO_URL = "https://github.com/HamzaMohamed25/flyrank-ml-internship"
PAPER_URL = "https://hamzamohamed25.github.io/flyrank-ml-internship/"
os.makedirs("docs/img", exist_ok=True); os.makedirs("work/figures", exist_ok=True)
LR, RF, RU = "Logistic Regression", "Random Forest", "Frozen rule"
A1, A2, A3, A4 = "1 new or thin history", "2 surge on an established page", "3 established, steady or rising", "4 established, dipping"
RATE = "next_month_decline_rate"
D_R, D_G, D_T, D_C = "random rows", "client-grouped", "time only", "client + time"

L6 = pd.DataFrame(w6["design_ladder"])
def lad(d, m, col="P@50"):
    r = L6[L6["design"].str.startswith(d) & L6["method"].str.startswith(m)]
    return float(r[col].iloc[0])
pf = pd.DataFrame(w5["per_fold"]); s5 = pf.groupby("method").agg(p10=("p10", "mean"), p50=("p50", "mean"), p100=("p100", "mean"), auc=("auc", "mean"), sd=("p50", "std"))
ev = w7["sealed_june_evaluation"]; EA = ev["all contract pages"]; EQ = next(v for k, v in ev.items() if k.startswith("queue"))
Adev, Ajun = pd.DataFrame(w7["archetypes_dev"]).set_index("archetype"), pd.DataFrame(w7["archetypes_june"]).set_index("archetype")
b_may, b_all, b_q = bl["base_rate"], EA["base_rate"], EQ["base_rate"]
n_ = lambda x: f"{int(round(x)):,}"; f2 = lambda x: f"{x:.2f}"; f3 = lambda x: f"{x:.3f}"; pc0 = lambda x: f"{100 * x:.0f}%"; pc1 = lambda x: f"{100 * x:.1f}%"
elig_jun = int(Ajun.loc[[A2, A3, A4], "pages"].sum()); thin_jun = int(Ajun.loc[A1, "pages"])

# The sentences below say "above", "below" or "no better" only if the numbers still do. If a re-run changes them, this stops and says which claim to revisit.
checks = {
 "random-row Random Forest beats its client-grouped score by more than 0.15": lad(D_R, RF) - lad(D_G, RF) > 0.15,
 "Logistic Regression is more than 0.10 above the base rate under client + time": lad(D_C, LR) - lad(D_C, LR, "base rate") > 0.10,
 "the frozen rule is not above the base rate under client + time": lad(D_C, RU) <= lad(D_C, RU, "base rate"),
 "June, queue population: Logistic Regression precision@50 is above the base rate": EQ[LR]["p50"] > EQ["base_rate"],
 "June, all pages: Logistic Regression precision@50 is not above the base rate": EA[LR]["p50"] <= EA["base_rate"],
 "June: the frozen rule is below the base rate": EQ[RU]["p50"] < EQ["base_rate"],
 "the base rate moved by more than 10 points from May to June": abs(b_all - b_may) > 0.10,
 "June: surge pages declined less often than steady pages": Ajun.loc[A2, RATE] < Ajun.loc[A3, RATE],
 "May: surge pages declined more often than steady pages": Adev.loc[A2, RATE] > Adev.loc[A3, RATE]}
bad = [k for k, v in checks.items() if not v]
assert not bad, "The paper text no longer matches the numbers. Revisit: " + "; ".join(bad)
print("Claim checks passed:", len(checks))

# ---- figures ----
INK, BLUE, ORANGE, GREY, GREYD = "#222222", "#1f6fb4", "#d9731a", "#8a8a8a", "#555555"
def clean(ax): ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=.25)
designs = [(D_R, "Random rows"), (D_G, "Client-grouped\nfolds"), (D_T, "Time only\n(Apr to May)"), (D_C, "Client + time\n(Apr to May)")]
x = np.arange(4); fig, ax = plt.subplots(figsize=(7.6, 4.4))
for m, col, ls, mk, lab in [(RF, ORANGE, "-", "s", "Random Forest"), (LR, BLUE, "-", "o", "Logistic Regression"), (RU, GREYD, "--", "^", "Frozen rule (ML-07)")]:
    ax.plot(x, [lad(d, m) for d, _ in designs], color=col, ls=ls, marker=mk, lw=2.4, ms=6, label=lab)
ax.plot(x, [lad(d, LR, "base rate") for d, _ in designs], color=GREY, lw=1.6, ls=":", marker="_", ms=16, label="Base rate (random picking)")
ax.annotate("Same model, same features:\nonly the split changed", xy=(1, lad(D_G, RF)), xytext=(1.3, lad(D_R, RF) - 0.01), fontsize=9, color=ORANGE, arrowprops=dict(arrowstyle="->", color=ORANGE))
ax.set_xticks(x, [l for _, l in designs]); ax.set_ylabel("precision@50"); ax.set_ylim(0.25, 0.8); clean(ax)
ax.set_title("A random split flatters the Random Forest; honest tests do not", loc="left", color=INK, fontsize=12)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, fontsize=9); fig.savefig("docs/img/fig1_design_ladder.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.2), sharey=True)
for ax, (title, e) in zip(axes, [("Queue population\n(thin-history pages excluded)", EQ), ("All contract pages", EA)]):
    ks = sorted(int(k) for k in e[LR]["curve"])
    ax.plot(ks, [e[LR]["curve"][str(k)] for k in ks], color=BLUE, lw=2.4, marker="o", ms=4, label="Logistic Regression")
    ax.plot(ks, [e[RU]["curve"][str(k)] for k in ks], color=ORANGE, lw=2.4, ls="--", marker="s", ms=4, label="Frozen rule (ML-07)")
    ax.axhline(e["base_rate"], color=GREY, lw=1.6, ls=":", label=f"base rate {e['base_rate']:.2f}"); ax.axvline(50, color=GREY, lw=1, alpha=.5)
    ax.set_title(title, loc="left", color=INK, fontsize=11); ax.set_xlabel("K: pages reviewed from the top of the list"); clean(ax)
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=1, fontsize=9)
axes[0].set_ylabel("share that declined in June (precision@K)"); fig.savefig("docs/img/fig2_precision_at_k_june.png", dpi=150, bbox_inches="tight"); plt.show()

labs = ["New or thin\nhistory", "Surge on an\nestablished page", "Steady or\nrising", "Dipping"]; idx = [A1, A2, A3, A4]; xx = np.arange(4); wd = 0.38
fig, ax = plt.subplots(figsize=(7.6, 4.2))
b1 = ax.bar(xx - wd / 2, [Adev.loc[a, RATE] for a in idx], wd, color=BLUE, label="May (next-month decline rate)"); b2 = ax.bar(xx + wd / 2, [Ajun.loc[a, RATE] for a in idx], wd, color=ORANGE, label="June (sealed month)")
for bars in (b1, b2): ax.bar_label(bars, labels=[f"{b.get_height():.2f}" for b in bars], padding=2, fontsize=9, color=INK)
ax.axhline(b_may, color=BLUE, ls="--", lw=1.4); ax.axhline(b_all, color=ORANGE, ls="--", lw=1.4)
ax.text(1.01, b_may, f"May base\nrate {b_may:.2f}", transform=ax.get_yaxis_transform(), color=BLUE, va="center", fontsize=9); ax.text(1.01, b_all, f"June base\nrate {b_all:.2f}", transform=ax.get_yaxis_transform(), color=ORANGE, va="center", fontsize=9)
ax.set_xticks(xx, labs); ax.set_ylim(0, 0.85); ax.set_ylabel("share that declined next month"); clean(ax)
ax.set_title("The month changed the base rate, and the order of the groups", loc="left", color=INK, fontsize=12)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, fontsize=9); fig.savefig("docs/img/fig3_archetype_rates.png", dpi=150, bbox_inches="tight"); plt.show()

mix = w7["queue_action_mix"]
vals = {"Verify spike, then review\n(in the queue)": mix.get("verify_spike_then_review", 0), "Investigate quiet risk\n(in the queue)": mix.get("investigate_quiet_risk", 0), "Monitor only\n(eligible, not in the queue)": elig_jun - sum(mix.values()), "Not on the queue\n(new or thin history)": thin_jun}
fig, ax = plt.subplots(figsize=(7.6, 3.6)); ks_, vs_ = list(vals)[::-1], list(vals.values())[::-1]
ax.barh(ks_, vs_, color=[ORANGE, BLUE, GREY, GREY]); [ax.text(v + max(vs_) * 0.01, i, f"{v:,}", va="center", fontsize=10, color=INK) for i, v in enumerate(vs_)]
ax.set_xlim(0, max(vs_) * 1.15); ax.set_xlabel("pages at the 1 June decision point"); ax.spines[["top", "right"]].set_visible(False)
ax.set_title("What the playbook does with every eligible page", loc="left", color=INK, fontsize=12); fig.savefig("docs/img/fig4_action_mix.png", dpi=150, bbox_inches="tight"); plt.show()
for f in os.listdir("docs/img"): shutil.copy(f"docs/img/{f}", f"work/figures/{f}")

# ---- html ----
def tbl(head, rows):
    th = "".join(f"<th scope='col'>{h}</th>" for h in head); tr = "".join("<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>" for r in rows)
    return f"<div class='tw'><table><thead><tr>{th}</tr></thead><tbody>{tr}</tbody></table></div>"
def figure(src, alt, cap): return f"<figure><img src='img/{src}' alt='{alt}' loading='lazy'><figcaption>{cap}</figcaption></figure>"
NB = REPO_URL + "/blob/main/work/notebooks/"
cov_html = ""
if COV:
    cov_html = "<p><strong>Coverage check.</strong> Days with data per month (of the calendar days): " + ", ".join(f"{m[5:]}/{m[:4]}: {v['days_with_data']}/{v['days_in_month']}" for m, v in COV.items()) + ". " + \
        ("Every month is complete." if all(v["days_with_data"] >= v["days_in_month"] for v in COV.values()) else "Months with fewer days than the calendar month: " + ", ".join(m for m, v in COV.items() if v["days_with_data"] < v["days_in_month"]) + ", so labels measured in those months can read low.") + "</p>"
dp_rows = [["1 April", "Jan-Mar", "April", n_(FACTS["apr_pages"]), FACTS["apr_clients"], f3(FACTS["apr_base"])],
           ["1 May", "Feb-Apr", "May", n_(bl["rows_analysis_frame"]), dc["clients"], f3(b_may)],
           ["1 June (sealed)", "Mar-May", "June", n_(w7["scoring_pages"]), FACTS["jun_clients"], f3(b_all)]]
lad_rows = [[lbl.replace("\n", " "), f2(lad(d, RU)), f"{f2(lad(d, LR))}", f"{f2(lad(d, RF))}", f3(lad(d, LR, "base rate"))] for d, lbl in designs]
s5_rows = [[m, f2(s5.loc[m, "p10"]), f"{f2(s5.loc[m, 'p50'])} &plusmn; {f2(s5.loc[m, 'sd'])}", f2(s5.loc[m, "p100"]), f3(s5.loc[m, "auc"])] for m in ["Frozen rule (ML-07)", "Logistic Regression", "Decision Tree (depth 4)", "Random Forest", "Hist Gradient Boosting"]]
jun_rows = [[name, n_(e["pages"]), f3(e["base_rate"]), f2(e[LR]["p10"]), f2(e[LR]["p50"]), f2(e[LR]["p100"]), f3(e[LR]["auc"]), f2(e[RU]["p50"])] for name, e in (("Queue population (thin-history pages excluded)", EQ), ("All contract pages", EA))]
arch_rows = [[a[2:].capitalize(), n_(Adev.loc[a, "pages"]), f3(Adev.loc[a, RATE]), n_(Ajun.loc[a, "pages"]), f3(Ajun.loc[a, RATE])] for a in idx]
abl = pd.DataFrame(w6["feature_ablation"]); abl_rows = [[v, f2(g[g.model == LR]["P@50"].iloc[0]), f3(g[g.model == LR]["AUC"].iloc[0]), f2(g[g.model == RF]["P@50"].iloc[0]), f3(g[g.model == RF]["AUC"].iloc[0])] for v, g in abl.groupby("variant", sort=False)]
str_rows = [[s["April clicks"], n_(s["pages"]), f3(s["base rate"]), f3(s["AUC within stratum"]), f3(s["precision in top 20% of scores"]), f"{s['lift (points)']:+.1f}"] for s in w6["size_strata"]]
trig_rows = [[t["check"], t["value"], "yes" if t["would fire"] else "no"] for t in w7["triggers"]]
gate_rows = [[g["no-go rule"], g["status"]] for g in w7["gates"]]
psi_rows = [[p["index"], f3(p[next(k for k in p if k.startswith("PSI"))]), p["status"]] for p in w7["psi"]]
mins = 10; hours = 50 * mins / 60; extra_q, extra_rule_q, extra_all = 50 * (EQ[LR]["p50"] - b_q), 50 * (EQ[RU]["p50"] - b_q), 50 * (EA[LR]["p50"] - b_all)
published = datetime.date.today().strftime("%d %B %Y")
base_ct = lad(D_C, LR, "base rate")

CSS = """:root{--bg:#fff;--fg:#1d1d1f;--muted:#565b63;--line:#d9dce1;--card:#f4f5f7;--accent:#1f5fa0;--warn:#8a4300;--warnbg:#fff3e0}
@media (prefers-color-scheme:dark){:root{--bg:#121417;--fg:#e8eaed;--muted:#a7adb6;--line:#2c3038;--card:#1b1e23;--accent:#7ab8f5;--warn:#ffc27a;--warnbg:#2a2113}}
*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;background:var(--bg);color:var(--fg);font:17px/1.65 system-ui,-apple-system,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
header,main,footer{max-width:780px;margin:0 auto;padding:0 20px}a{color:var(--accent)}
.hero{padding-top:48px}.eyebrow{font-size:13px;letter-spacing:.06em;text-transform:uppercase;color:var(--muted);margin:0 0 8px}
h1{font-size:2.1rem;line-height:1.2;margin:.1em 0 .3em}.sub{font-size:1.1rem;color:var(--muted);margin:0 0 .6em}.meta{font-size:14px;color:var(--muted)}
h2{font-size:1.5rem;margin:2.2em 0 .5em;padding-top:.4em;border-top:1px solid var(--line)}h3{font-size:1.15rem;margin:1.6em 0 .4em}
.tiles{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px;margin:1.2em 0}.tile{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:14px 16px}
.tile b{display:block;font-size:1.7rem;line-height:1.2}.tile span{font-size:14px;color:var(--muted)}
.tldr{background:var(--card);border:1px solid var(--line);border-left:5px solid var(--accent);border-radius:8px;padding:12px 18px;margin:1.2em 0}.tldr p{margin:.35em 0}
.warn{background:var(--warnbg);border-left:5px solid var(--warn);border-radius:8px;padding:10px 16px;margin:1.2em 0;color:var(--fg)}
nav ol{columns:2;font-size:15px;padding-left:1.2em}.tw{overflow-x:auto;margin:1em 0}table{border-collapse:collapse;width:100%;font-size:15px}
th,td{border-bottom:1px solid var(--line);padding:7px 10px;text-align:left;vertical-align:top}th{background:var(--card)}td:not(:first-child){font-variant-numeric:tabular-nums}
figure{margin:1.6em 0;background:#fff;border:1px solid var(--line);border-radius:10px;padding:10px}figure img{display:block;margin:0 auto;max-width:100%;height:auto}
figcaption{font-size:14px;color:#444;margin-top:8px}code{background:var(--card);padding:1px 5px;border-radius:4px;font-size:.92em}
footer{padding:2em 20px 3em;color:var(--muted);font-size:14px;border-top:1px solid var(--line);margin-top:3em}@media (max-width:560px){nav ol{columns:1}h1{font-size:1.7rem}}"""

html = f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Which pages should a content team review first? A client-grouped, time-forward test of a refresh queue</title>
<meta name="description" content="An honest test of a content-refresh ranking on {n_(dc['rows'])} search pages from {dc['clients']} clients: client-grouped folds, a time-forward check and a sealed month. Observed, directional, decision-support.">
<style>{CSS}</style></head><body>
<header class="hero"><p class="eyebrow">Applied ML research note &middot; Refresh / content opportunity scoring</p>
<h1>Which pages should a content team review first?</h1>
<p class="sub">A client-grouped, time-forward and sealed-month test of a content-refresh queue on {n_(dc['rows'])} pages from {dc['clients']} clients</p>
<p class="meta">By {AUTHOR_NAME} &middot; FlyRank ML Internship capstone &middot; Published {published} &middot; Data: January to June 2026 &middot; <a href="{REPO_URL}">Code and receipts</a></p></header>
<main>
<nav aria-label="Contents"><ol><li><a href="#abstract">Abstract</a></li><li><a href="#introduction">Introduction</a></li><li><a href="#data">Data</a></li><li><a href="#methodology">Methodology</a></li><li><a href="#results">Results</a></li><li><a href="#limitations">Limitations</a></li><li><a href="#recommendations">Ranked recommendations</a></li><li><a href="#reproducibility">Reproducibility</a></li><li><a href="#acknowledgments">Acknowledgments</a></li></ol></nav>

<section id="abstract"><h2>Abstract</h2>
<p>A content team can review about 50 pages a month out of thousands, so the useful question is which 50 to open first. Using daily clicks, impressions and position for {n_(dc['rows'])} eligible pages from {dc['clients']} clients (FlyRank ML Internship dataset, January to June 2026), I labelled a page as declining when its next-month clicks fell below 80% of the latest month, and compared a frozen, transparent rule with a Logistic Regression and a Random Forest. Validation was client-grouped, then time-forward, and it ended with a sealed month (June 2026) that was read once, after every setting was frozen. A random split flattered the Random Forest (precision@50 of {f2(lad(D_R, RF))}), but on unseen clients and a later month the two models scored {f2(lad(D_C, LR))} and {f2(lad(D_C, RF))} against a {f2(base_ct)} base rate and {f2(lad(D_C, RU))} for the rule; in the sealed June month, when the share of declining pages jumped from {f2(b_may)} to {f2(b_all)}, the model's 50-page queue scored {f2(EQ[LR]['p50'])} against {f2(b_q)} (but {f2(EA[LR]['p50'])} against {f2(b_all)} across all pages) and the rule fell below random. The evidence supports using the score to order a human review under monthly monitoring, not as a detector of content decay, and it supports no causal claim about refreshing.</p>
<div class="tiles"><div class="tile"><b>{f2(lad(D_R, RF))} &rarr; {f2(lad(D_G, RF))}</b><span>Random Forest precision@50: random rows vs unseen clients (same model, same features)</span></div>
<div class="tile"><b>{f2(lad(D_C, LR))} vs {f2(base_ct)}</b><span>Logistic Regression top-50 hit rate vs random picking, unseen clients and a later month (April to May)</span></div>
<div class="tile"><b>{f2(EQ[LR]['p50'])} vs {f2(b_q)}</b><span>Sealed June month, queue top-50 vs base rate. Across all pages: {f2(EA[LR]['p50'])} vs {f2(b_all)}</span></div></div>
<aside class="tldr" aria-label="Summary"><p><strong>Question:</strong> which 50 pages should a person review first this month?</p><p><strong>Data:</strong> {n_(dc['rows'])} eligible pages, {dc['clients']} clients, three decision points, January to June 2026.</p><p><strong>Result:</strong> modest and unstable lift over random picking; a transparent rule that did not hold in the sealed month.</p><p><strong>Decision it supports:</strong> ordering a human review, with monthly checks.</p><p><strong>Biggest limitation:</strong> the label rate swung from {f2(b_may)} to {f2(b_all)} between two months, and the ranking is weak overall (AUC about {f2(lad(D_C, LR, 'AUC'))}).</p></aside></section>

<section id="introduction"><h2>Introduction</h2>
<p>Every month a content strategist faces thousands of published pages and time to look at a few dozen. The decision this work supports is <strong>which 50 pages to review first</strong>. That makes it a ranking problem judged at a fixed review budget: of the 50 pages on the list, how many really lost clicks the following month? Because a hit rate means little on its own, every number below is shown next to the <strong>base rate</strong>, the share you would get by picking pages at random.</p>
<p>The target is an observed proxy, not a business outcome. A page is <em>declining</em> when its next-month clicks are below 80% of its latest month. The proxy does not say why a page declined and does not say that a refresh would help. Language in this paper is deliberately limited to what was <em>observed</em> and <em>measured</em>, what is <em>directional</em>, and what is <em>decision-support</em>.</p>
<p>The question belongs to the lane &ldquo;refresh / content opportunity scoring&rdquo;. The paper reports what held up, what did not, and one sealed month that reversed part of the story.</p></section>

<section id="data"><h2>Data</h2>
<p><strong>Release and tables.</strong> FlyRank ML Internship warehouse, release {dc['release']}. Only the daily page-performance table was used, and only three measures from it: clicks, impressions and average position. Daily rows were summed into page-level months. Page and client identifiers are hashed, and nothing here names a client, domain, URL or query.</p>
<p><strong>Windows.</strong> Each decision point looks back over three months and forward one. Pages are counted after the identical-row block (below) is excluded.</p>
{tbl(["Decision point", "Features from", "Label month", "Pages", "Clients", "Share declining (base rate)"], dp_rows)}
<p><strong>Eligibility.</strong> A page enters when it has at least 1,000 impressions over the three feature months and at least 10 clicks in the latest one. At the 1 May point that leaves {n_(dc['rows'])} of {n_(dc['rows_before_eligibility'])} pages from {dc['clients']} clients ({n_(bl['rows_analysis_frame'])} after the exclusion below).</p>
<p><strong>What was left out, and why.</strong></p><ul>
<li>Product flags, health scores and trend labels were never loaded: they risk leaking the answer and are not public-safe.</li>
<li>Session, scroll and AI-referral fields were excluded: they exist only for some pages, and a zero before tracking began does not mean nothing happened.</li>
<li>Identifiers are hashed and used only to join and to keep each client on one side of a split.</li>
<li>A block of {n_(bl['templated_block']['rows_excluded'])} pages from one client shares an identical feature vector, and {pc1(bl['templated_block']['decline_rate'])} of them declined. Identical rows are not independent evidence, so any feature vector repeated 20 or more times is excluded ({n_(w7['identical_block_excluded']['scoring'])} pages at the June point). The rule uses feature columns only.</li>
<li>Pages with fewer than 10 clicks in the first or middle month (new or thin history) are scored but are not eligible for the queue.</li></ul>{cov_html}</section>

<section id="methodology"><h2>Methodology</h2>
<h3>Label, features and baseline</h3>
<p>The label is 1 when next-month clicks are below 80% of the latest month. The nine features are known at the decision point: impressions and clicks over the window, impressions and clicks in the latest month, clicks in the first month, a log-momentum term, CTR, an impression-weighted position, and the number of active days. Counts are log-transformed and scaled for the Logistic Regression.</p>
<p>The <strong>frozen baseline rule</strong> flags a page when its latest-month clicks are at least {bl['min_ratio']:.0f}&times; the average of the two months before, and the page had at least {bl['history_floor']} clicks in each of those two months. Flagged pages are ordered by surge strength times impressions percentile. Exact ties are broken by a seeded hash, never by file order. The rule was written and frozen before any model was trained.</p>
<h3>Models and metric</h3>
<p>A Logistic Regression (the readable first model), a depth-4 Decision Tree, a Random Forest (300 trees, declared as the primary model before any result) and a small fixed Hist Gradient Boosting, all with fixed settings and no tuning on any held-out fold. The metric is precision@50, the review budget, with precision@10, @100 and AUC as secondary. The base rate sits beside every result. Every method sees the same pages, folds, metric, K and tie policy. Seed {w5['seed']}.</p>
<h3>Validation design</h3>
<p>Four designs, in order of how honest they are. <strong>Random rows</strong> (the contrast): pages of the same client are on both sides. <strong>Client-grouped folds</strong>: five folds, every client in exactly one test fold, asserted in code. <strong>Time only</strong>: train at the 1 April point, test at 1 May without changing anything. <strong>Client + time</strong>: train on April pages of other clients, test on May pages of the held-out clients. Finally, a <strong>sealed month</strong>: the model is trained only at 1 May, the queue is built at 1 June from March to May, and June is read once, after the settings, thresholds and archetype cut-offs are printed and frozen.</p>
<h3>Leakage controls</h3>
<p>Features end before the label window starts. The label's ingredient (next-month clicks) is never a feature. As a harness, I added a column built from it on purpose: precision@50 went to {f2(w6['leak_harness']['P@50'])} and AUC to {f3(w6['leak_harness']['AUC'])}, so the audit can see a leak; the column was removed. Groups of features were also removed to see whether the result depends on a suspect group.</p>
<div class="warn"><strong>Choices made after looking at results, disclosed.</strong> The rule direction was chosen after a signal check (the first idea, &ldquo;keeps falling, so refresh&rdquo;, pointed the wrong way). A history guard was added after the weak picks of a first version, and the identical-row block was excluded after diagnostics. The Logistic Regression was highlighted after seeing five folds, which adds some optimism, and this is why a sealed month was added.</div></section>

<section id="results"><h2>Results</h2>
<h3>1. The more honest the test, the lower the score</h3>
{tbl(["Validation design", "Frozen rule", "Logistic Regression", "Random Forest", "Base rate"], lad_rows)}
{figure("fig1_design_ladder.png", "Line chart of precision at 50 for the rule, Logistic Regression and Random Forest under four validation designs, with the base rate", f"<strong>Figure 1.</strong> Precision@50 by validation design. <em>What to notice:</em> the Random Forest drops from {f2(lad(D_R, RF))} with random rows to {f2(lad(D_G, RF))} with client-grouped folds. The Logistic Regression barely moves, because a low-capacity model cannot memorize clients. Under the strictest design both models sit about {100 * (lad(D_C, LR) - base_ct):.0f} to {100 * (lad(D_C, RF) - base_ct):.0f} points above the base rate, and the rule does not.")}
<p>Roughly {100 * (lad(D_R, RF) - lad(D_G, RF)):.0f} points of the Random Forest's random-row score came from recognizing clients it had already seen. Under client + time, the Logistic Regression ({f2(lad(D_C, LR))} &plusmn; {f2(lad(D_C, LR, 'P@50 sd across folds'))}) and the Random Forest ({f2(lad(D_C, RF))} &plusmn; {f2(lad(D_C, RF, 'P@50 sd across folds'))}) are indistinguishable at this size. The base rate itself moved from {f3(FACTS['apr_base'])} at the April point to {f3(b_may)} at the May point.</p>
<h3>2. Five methods on client-grouped folds</h3>
{tbl(["Method", "P@10", "P@50 (mean &plusmn; sd over folds)", "P@100", "AUC"], s5_rows)}
<p>Base rate {f3(pf.groupby('fold')['base_rate'].first().mean())}. The two largest clients each form a fold of their own, so two of the five folds measure a single client. The Logistic Regression was best and the Random Forest, declared primary in advance, was the least stable (in one fold it fell below random). All AUC values are near 0.57 to 0.58, so the ranking is weak overall and useful mostly at the head of the list.</p>
<h3>3. The sealed month</h3>
{tbl(["Population", "Pages", "Base rate", "Model P@10", "Model P@50", "Model P@100", "Model AUC", "Rule P@50"], jun_rows)}
{figure("fig2_precision_at_k_june.png", "Two panels of precision at K for June: Logistic Regression and the frozen rule against the base rate, for the queue population and for all contract pages", "<strong>Figure 2.</strong> Precision@K in the sealed June month. <em>What to notice:</em> on the queue population (left) the model sits above the base rate along the whole list. Across all contract pages (right) it is below the base rate at the review budget of 50 and moves slightly above it only further down the list. The dotted line is random picking. The vertical line marks the review budget of 50.")}
<p>The result depends on the population. Among pages eligible for the queue, the model's top 50 declined {pc0(EQ[LR]['p50'])} of the time against a {pc0(b_q)} base rate. With only 50 pages, the chance margin is about &plusmn;{100 * 1.96 * (EQ[LR]['p50'] * (1 - EQ[LR]['p50']) / 50) ** 0.5:.0f} points, so this is directional. Across all contract pages the model's top 50 declined {pc0(EA[LR]['p50'])} against {pc0(b_all)}. AUC is {f3(EQ[LR]['auc'])} on the queue population and {f3(EA[LR]['auc'])} on all pages. The frozen rule's top 50 declined {pc0(EQ[RU]['p50'])} of the time, below the base rate.</p>
{figure("fig3_archetype_rates.png", "Bar chart of next-month decline rate by page archetype in May and in June with the base rate of each month", f"<strong>Figure 3.</strong> Next-month decline rate by archetype. <em>What to notice:</em> the share of declining pages rose from {f2(b_may)} to {f2(b_all)}, and the order of the groups changed. In May, surge pages declined more often than steady ones ({f2(Adev.loc[A2, RATE])} vs {f2(Adev.loc[A3, RATE])}); in June they declined less often ({f2(Ajun.loc[A2, RATE])} vs {f2(Ajun.loc[A3, RATE])}).")}
{tbl(["Archetype", "May pages", "May decline rate", "June pages", "June decline rate"], arch_rows)}
<p>The idea that &ldquo;a spike reverts&rdquo; therefore held weakly in May and did not hold in June. The monitoring check on the base rate fired, as it was designed to.</p>
<h3>4. What the leakage audit found</h3>
{tbl(["Features used (client-grouped, base rate 0.391)", "LR P@50", "LR AUC", "RF P@50", "RF AUC"], abl_rows)}
<p>Click-level features are legal inputs, because they are known before the label month, but they sit close to the label and the result leans on them: without them the Logistic Regression falls to {f2(float(abl[(abl.variant.str.startswith('without')) & (abl.model == LR)]['P@50'].iloc[0]))} (still above the base rate). Click-level features alone do not work, so the signal comes from clicks and impressions together.</p>
{tbl(["April clicks", "Pages", "Base rate", "AUC inside the band", "Precision in the top 20% of scores", "Lift (points)"], str_rows)}
<p>Small pages decline more often, which fits regression to the mean. Inside each size band the broad ranking adds little, so a good part of the overall ordering is sorting by page size. This band check looks at the top 20% of scores, not at the top 50.</p>
<h3>5. What the errors look like</h3>
<p>In the five client-grouped top-50 lists of the Random Forest, {FACTS['err_hits']} of {FACTS['err_total']} pages declined ({pc0(FACTS['err_hits'] / FACTS['err_total'])}, against a 39% base rate), but the median hit and miss were almost identical (14 April clicks, 2 February clicks, 71 active days). The lists are full of small pages near the 10-click floor, where one or two clicks decide the label. The declines the model missed were bigger and steadier pages: {n_(FACTS['err_missed_low'])} of {n_(FACTS['err_declines'])} declined pages scored in the bottom quarter of their fold, against {n_(FACTS['err_top_quarter'])} in the top quarter, while chance would give a quarter each. The model catches small pages that fall back and misses established pages that decline, and the second kind is closer to decay.</p></section>

<section id="limitations"><h2>Limitations and honest framing</h2>
<ul><li><strong>Three windows, one month each.</strong> The forward evidence is the April-to-May test and the sealed May-to-June month. That is thin temporal evidence, not a walk-forward over many months.</li>
<li><strong>The ranking is weak overall.</strong> AUC is {f2(L6['AUC'].min())} to {f2(L6['AUC'].max())} across designs (the rule is about 0.50), and the spread across folds is 7 to 13 points at precision@50.</li>
<li><strong>The label rate moved a lot.</strong> It was {f2(FACTS['apr_base'])}, {f2(b_may)} and {f2(b_all)} at the three decision points. This work does not establish whether the June jump is seasonal, real or a data artifact{(' (the coverage check above reports the days present per month)' if COV else '')}. The scores order pages; they are not calibrated probabilities, and calibration was not evaluated.</li>
<li><strong>The population changes the conclusion.</strong> On queue-eligible pages the model is above the base rate in June; on all pages it is not.</li>
<li><strong>Click-level features.</strong> The result leans on features close to the label, and the broad ranking adds little inside page-size bands.</li>
<li><strong>The label is a proxy</strong> and partly reflects regression to the mean. It cannot see slow decay over several months.</li>
<li><strong>The portfolio is lopsided:</strong> the top five clients hold 72% of the pages, and client base rates run from 3% to 75%.</li>
<li><strong>The queue is made of small pages:</strong> median {FACTS['queue_median_clicks']} clicks in the latest month, {w7['queue_confidence'].get('low', 0)} of {sum(w7['queue_confidence'].values())} rows carry a low-confidence note, and only {w7['queue_action_mix'].get('verify_spike_then_review', 0)} of {sum(w7['queue_action_mix'].values())} rows has a rule-based reason code.</li>
<li><strong>No causal claim.</strong> Nothing here shows that refreshing a page recovers traffic, and nothing claims to explain a search engine's ranking.</li></ul></section>

<section id="recommendations"><h2>Ranked recommendations (action playbook)</h2>
<p>The playbook was built to be used by a person: <strong>the rule supplies the reason, the model supplies the order, and a human decides.</strong> It is a plan with limits, not a production system.</p>
{tbl(["Action", "When", "Reason code"], [["Verify the spike, then review", "The rule flags the page (latest month at least 2x the earlier average, with enough history)", "click_surge_vs_prior"], ["Investigate quiet risk", "The model puts the page in the top 50 but the rule has no flag", "model_score_high_no_rule_flag"], ["Monitor only", "Established pages not in the queue", "none"], ["Not on the queue", "Fewer than 10 clicks in the first or middle month", "thin_history"]])}
{figure("fig4_action_mix.png", "Horizontal bar chart of how many pages fall into each playbook action at the 1 June decision point", f"<strong>Figure 4.</strong> What the playbook does with the {n_(w7['scoring_pages'])} pages at the 1 June decision point. <em>What to notice:</em> a 50-page queue is a small slice, and {n_(thin_jun)} pages are not eligible at all.")}
<h3>Before acting on any row, a person checks</h3>
<ol><li>Is the spike a tracking or campaign artifact (always, above 7x)?</li><li>Is the topic seasonal?</li><li>Is it a durable climb that was already rising?</li><li>Is the page still about what people search for?</li><li>Is the client changing or recently optimizing the page?</li><li>Are the counts tiny (fewer than 30 clicks)? One or two clicks can decide the label.</li></ol>
<h3>What must not be automated</h3>
{tbl(["No-go rule", "Status in this work"], gate_rows)}
<p>The queue never authorizes a rewrite, and a model score alone is never a reason. Scores are not shown as calibrated probabilities.</p>
<h3>Monitoring and retraining triggers</h3>
<p>A light plan with two clocks: checks before the queue is released (population size, feature drift) and checks after labels arrive (base rate, model against rule and base rate). These are proposed thresholds, run here in a notebook only. This is what they showed on the sealed June month:</p>
{tbl(["Check", "Value", "Would fire"], trig_rows)}
{tbl(["Feature", "PSI (1 May training vs 1 June scoring)", "Status"], psi_rows)}
<h3>Cost and value, with the assumptions visible</h3>
<p>The value of a batch is the number of pages that really declined in the list beyond random picking: 50 &times; (precision@50 &minus; base rate). On queue-eligible pages in June this is {extra_q:+.1f} pages for the model and {extra_rule_q:+.1f} for the rule; across all pages it is {extra_all:+.1f} for the model. The cost is reviewer time. At an <em>assumed</em> {mins} minutes per page, a batch takes {hours:.1f} hours. This is a descriptive scenario from one month, not a forecast, and it says nothing about traffic recovery, which would need a controlled test.</p>
<h3>What I would do next</h3>
<ul><li>Record what reviewers decide for each row, to build outcome labels that are not a proxy.</li><li>Run the same test over many more months (a walk-forward) and check why the label rate jumped in June.</li><li>Test whether the top-50 lists sit near the 10-click floor, and add a longer window for slow decay.</li><li>Calibrate the scores and check each client separately.</li></ul></section>

<section id="reproducibility"><h2>Reproducibility</h2>
<ul><li><strong>Repository:</strong> <a href="{REPO_URL}">{REPO_URL.replace('https://', '')}</a>. Notebooks, in order: <a href="{NB}w03_data_contract.ipynb">data contract</a>, <a href="{NB}w04_baseline_score.ipynb">baseline</a>, <a href="{NB}w05_model.ipynb">model</a>, <a href="{NB}w06_validation_audit.ipynb">validation audit</a>, <a href="{NB}w07_action_playbook.ipynb">action playbook</a> and <a href="{NB}capstone.ipynb">this capstone notebook</a>.</li>
<li><strong>Receipts:</strong> every number on this page comes from the committed JSON files in <code>work/outputs/</code> (data contract, baseline, model, validation audit, playbook). The capstone notebook reads them and regenerates this page and its figures.</li>
<li><strong>Sealed month:</strong> the cell that builds the June frame and the file it produced (<code>w07_playbook_metrics.json</code>) are both committed. Settings were printed before June was read.</li>
<li><strong>Settings:</strong> seed {w5['seed']}; Python {w5['versions']['python']}, NumPy {w5['versions']['numpy']}, pandas {w5['versions']['pandas']}, scikit-learn {w5['versions']['scikit-learn']}. Data release {dc['release']} (gated; access is requested from the program).</li>
<li><strong>Not committed, by design:</strong> data files and the page-level queue. They are regenerated by the notebooks.</li></ul></section>

<section id="acknowledgments"><h2>Acknowledgments and data credit</h2>
<p><strong>Built on the FlyRank ML Internship dataset</strong> (<a href="https://flyrank.ai" target="_blank" rel="noopener">https://flyrank.ai</a>). Thanks to the FlyRank internship team for the live sessions and the framework this work follows. The research paper &ldquo;The State of AI-Driven SEO, March 2026&rdquo; (FlyRank) was used as the example in the claim audit. Models: scikit-learn. This is student work; it makes no claim about how any search engine ranks pages.</p></section>
</main>
<footer>Generated from committed receipts by <code>work/notebooks/capstone.ipynb</code> on {published}. Observed, measured, directional, decision-support.</footer></body></html>"""
open("docs/index.html", "w", encoding="utf-8").write(html)
open("docs/.nojekyll", "w").write("")
shutil.make_archive("paper_site", "zip", "docs")
print(f"Wrote docs/index.html ({len(html):,} characters) and 4 figures. Zipped as paper_site.zip.")
try:
    from google.colab import files; files.download("paper_site.zip")
except Exception:
    pass

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [✓] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [✓] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
